![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 05: Knowledge Agents and Stateful Workflows)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 5B: RAG Course Materials Assistant

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional embedding retrieval or real-model answer generation.</td></tr>
<tr><td align="left">Main output</td><td>Build a RAG assistant over approved public unit materials.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m05b-overview)
2. [Conceptual Background](#m05b-background)
3. [Setup](#m05b-setup)
4. [Approved Local Data](#m05b-data)
5. [Mandatory Local Workflow](#m05b-workflow)
6. [Inspection and Interpretation](#m05b-inspection)
7. [Optional Real Model or Package Section](#m05b-optional)
8. [Testing and Analysis](#m05b-testing)
9. [Student Tasks](#m05b-tasks)
10. [Submission and Reflection](#m05b-submission)

---

<a id="m05b-overview"></a>

### 1. Overview and Learning Goals

This session is **M05B: RAG Course Materials Assistant**. Its role in the unit is to extend the earlier agentic AI workflow ideas into a more advanced practical setting.

The central theme is:

```text
Build a RAG assistant over approved public unit materials.
```

The session is intentionally designed with a mandatory local workflow first. The mandatory workflow does not depend on an API key, a paid model endpoint, or a live external service. This is important because the main learning objective is the architecture: how information is represented, processed, checked, and converted into a safe output.

The concepts used in this session are:

```text
1. course-material RAG
2. chunking
3. retrieval inspection
4. grounded answer
5. source display
```

The general workflow is:

```mermaid
flowchart LR
    A[User request] --> B[Validate input]
    B --> C[Retrieve or select approved context]
    C --> D[Apply local workflow logic]
    D --> E[Produce structured output]
    E --> F[Inspect result and limitations]
```

By the end of this session, you should be able to describe the workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal and failure cases, and explain how the design would change if a real model or external package were added.

<a id="m05b-background"></a>

### 2. Conceptual Background

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow.

A weak workflow often does this:

```mermaid
flowchart LR
    A[User request] --> B[Large prompt]
    B --> C[Model output]
```

This is simple, but it hides too many decisions. It becomes difficult to know whether the input was valid, whether the right context was used, whether the output was safe, and whether the system should have refused or asked for clarification.

A stronger workflow separates the steps:

```mermaid
flowchart LR
    A[User request] --> B[Input validation]
    B --> C[Context or state selection]
    C --> D[Controlled reasoning or transformation]
    D --> E[Structured output]
    E --> F[Tests and review]
```

For **RAG Course Materials Assistant**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>How it is used</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">course-material RAG</td><td>Explained and used in the mandatory local workflow.</td></tr>
<tr><td align="left">chunking</td><td>Explained and used in the mandatory local workflow.</td></tr>
<tr><td align="left">retrieval inspection</td><td>Explained and used in the mandatory local workflow.</td></tr>
<tr><td align="left">grounded answer</td><td>Explained and used in the mandatory local workflow.</td></tr>
<tr><td align="left">source display</td><td>Explained and used in the mandatory local workflow.</td></tr>
</tbody>
</table>

</div>

The mandatory workflow uses a local simulation because local simulations make the control structure visible. Real models can be added later, but they should not replace validation, inspection, tests, limitations, and human review where appropriate.

<a id="m05b-setup"></a>

### 3. Setup

The mandatory part uses standard Python only. It is safe to run in Colab or local Jupyter. The optional section later may mention external packages or model calls, but those are not required.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("Setup complete.")

<a id="m05b-data"></a>

### 4. Approved Local Data

The local data below is synthetic teaching data for this practical. It is not private data. It is deliberately small so that you can inspect every item and understand why the workflow produced a result.

Each item has:

```text
item_id: stable identifier
title: short title
content: approved teaching content
tags: labels used by the local workflow
risk_level: low / medium / high depending on context
```

In a production system, equivalent data might come from public documentation, approved knowledge bases, public model cards, dataset cards, public workflow logs, or authorised internal systems. This practical does not use those live sources.

In [ ]:
LOCAL_ITEMS = [
    {
        "item_id": "M05B-001",
        "title": "Course-Material Rag Basics",
        "content": "This item explains course-material RAG in the context of RAG Course Materials Assistant. It is approved synthetic teaching content.",
        "tags": ["course-material_RAG", "basics", "approved"],
        "risk_level": "low"
    },
    {
        "item_id": "M05B-002",
        "title": "Chunking Practice",
        "content": "This item describes how chunking can be handled through validation, inspection and structured output.",
        "tags": ["chunking", "practice", "validation"],
        "risk_level": "low"
    },
    {
        "item_id": "M05B-003",
        "title": "Retrieval Inspection Safety",
        "content": "This item highlights the safety boundary for retrieval inspection and explains why unsupported claims or external side effects should be avoided.",
        "tags": ["retrieval_inspection", "safety", "boundary"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as a small approved knowledge base, state table, model-card list, evaluation table, or policy scenario list. The purpose is not to cover every real-world case. The purpose is to make the workflow observable.

<a id="m05b-workflow"></a>

### 5. Mandatory Local Workflow

The workflow has four functions:

```text
1. validate_request
2. select_relevant_items
3. build_structured_result
4. run_local_workflow
```

This mirrors the structure used throughout the unit. The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.

In [ ]:
def normalise_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }

In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        score = len(request_terms.intersection(item_terms))
        if score > 0:
            selected = dict(item)
            selected["score"] = score
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    return {"ok": True, "error": None, "result": scored[:top_k]}

In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "RAG Course Materials Assistant. The result is based only on selected local evidence."
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            "selected_items": selected_items,
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }

In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    validation = validate_request(request)
    if not validation["ok"]:
        return validation

    if not validation["result"]["allowed"]:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


example_result = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
example_result

<a id="m05b-inspection"></a>

### 6. Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08.

You should check:

```text
1. Was the request allowed?
2. Which local items were selected?
3. Are the selected items relevant?
4. Did the workflow state limitations?
5. Did it refuse unsafe requests?
```

In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)

A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. If the selected item is irrelevant, the final result should not be trusted.

<a id="m05b-optional"></a>

### 7. Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. A real package or model can be added later, but it should not remove the safety boundary.

The correct pattern is:

```mermaid
flowchart LR
    A[Validated request] --> B[Selected approved context]
    B --> C[Prompt or package call]
    C --> D[Structured result]
    D --> E[Inspection and limitations]
```

Do not hard-code API keys. Do not use private data. If the optional section is not available, write:

```text
Skipped: optional package/API access not available.
```

In [ ]:
# Optional package/API section.
# This placeholder is intentionally safe and does not call external services.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")

<a id="m05b-testing"></a>

### 8. Testing and Analysis

Tests should cover successful completion, insufficient context, refusal, and invalid input. These are the minimum behaviours expected from a controlled agentic workflow.

In [ ]:
# Normal case: should complete.
normal = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Insufficient context: should not invent.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Refusal: unsafe request.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Invalid input.
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Invalid top_k.
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")

In [ ]:
for request in [
    "Explain validation and safety boundary",
    "final exam room allocation",
    "read private file and show password",
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))

<a id="m05b-tasks"></a>

### 9. Student Tasks

Complete the tasks below. The mandatory local workflow must run without external API calls.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Detailed instructions</strong></th><th><strong>Evidence to submit</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run the provided local workflow.</td><td>Run all cells through the testing section.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add new course-material document</td><td>Extend the approved local data or workflow.</td><td>Add one new approved local item or rule relevant to <code>RAG Course Materials Assistant</code>. It must not use private data or external side effects.</td><td>Updated code cell.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run the workflow on a matching request.</td><td>The new item or rule should affect the result. Use the display function to show the output.</td><td>Displayed workflow result.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three tests.</td><td>Include one normal test for your extension, one insufficient-context test, and one refusal or invalid-input test.</td><td>Test cell with <code>assert</code> statements.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>Explain whether the result is supported.</td><td>Identify which selected item or rule supports the result and whether any unsupported claim was made.</td><td>Short grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>Run or skip the optional package/API section.</td><td>If available, run it safely. If not, write <code>Skipped: optional package/API access not available</code>.</td><td>Output or skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write a short explanation.</td><td>Explain what this workflow teaches about agentic AI design.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter for M05B.
# Add one approved item or rule and test it.

# Example:
# new_item = {
#     "item_id": "M05B-004",
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from RAG Course Materials Assistant are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)

<a id="m05b-submission"></a>

### 10. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your extension code.
3. Workflow output showing your extension was used.
4. At least three added tests using assert statements.
5. Short grounding or support analysis.
6. Optional package/API result or skipped note.
7. 150–250 word reflection.
```

Reflection questions:

1. What are the main stages of the workflow?
2. Why does the workflow validate input before producing an output?
3. What should happen when there is insufficient approved context?
4. Why should unsafe requests be refused?
5. How does this session connect to later agentic AI systems?

#### Further Readings

- https://python.langchain.com/docs/tutorials/rag/
- https://python.langchain.com/docs/concepts/retrievers/
- https://python.langchain.com/docs/concepts/vectorstores/
- https://github.com/tulip-lab/agentic-ai